### Import Libraries

In [1]:
import pandas as pd
import numpy as np

# Display settings
pd.set_option('display.max_columns', None)


### Load Dataset

In [4]:
import os

BASE_DIR = os.path.abspath(".")

file_path = os.path.join(BASE_DIR, "../data/raw/market_data/give_some_credit.csv")

df = pd.read_csv(file_path)

df.head()

,Unnamed: 0,target_default,credit_utilization,age,late_30_59_days,debt_ratio,monthly_income,total_credit_lines,late_90_days,real_estate_loans,late_60_89_days,dependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [5]:
df.info()
df.describe()


<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Unnamed: 0          150000 non-null  int64  
 1   target_default      150000 non-null  int64  
 2   credit_utilization  150000 non-null  float64
 3   age                 150000 non-null  int64  
 4   late_30_59_days     150000 non-null  int64  
 5   debt_ratio          150000 non-null  float64
 6   monthly_income      120269 non-null  float64
 7   total_credit_lines  150000 non-null  int64  
 8   late_90_days        150000 non-null  int64  
 9   real_estate_loans   150000 non-null  int64  
 10  late_60_89_days     150000 non-null  int64  
 11  dependents          146076 non-null  float64
dtypes: float64(4), int64(8)
memory usage: 13.7 MB


,Unnamed: 0,target_default,credit_utilization,age,late_30_59_days,debt_ratio,monthly_income,total_credit_lines,late_90_days,real_estate_loans,late_60_89_days,dependents
count,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,1.202690e+05,150000.000000,150000.000000,150000.000000,150000.000000,146076.000000
mean,75000.500000,0.066840,6.048438,52.295207,0.421033,353.005076,6.670221e+03,8.452760,0.265973,1.018240,0.240387,0.757222
std,43301.414527,0.249746,249.755371,14.771866,4.192781,2037.818523,1.438467e+04,5.145951,4.169304,1.129771,4.155179,1.115086
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,37500.750000,0.000000,0.029867,41.000000,0.000000,0.175074,3.400000e+03,5.000000,0.000000,0.000000,0.000000,0.000000
50%,75000.500000,0.000000,0.154181,52.000000,0.000000,0.366508,5.400000e+03,8.000000,0.000000,1.000000,0.000000,0.000000
75%,112500.250000,0.000000,0.559046,63.000000,0.000000,0.868254,8.249000e+03,11.000000,0.000000,2.000000,0.000000,1.000000
max,150000.000000,1.000000,50708.000000,109.000000,98.000000,329664.000000,3.008750e+06,58.000000,98.000000,54.000000,98.000000,20.000000


In [7]:
# Replace "NA" with NaN
df.replace("NA", np.nan, inplace=True)

# Convert only object columns to numeric where possible
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')


# Check missing values
df.isnull().sum()


Unnamed: 0                0
target_default            0
credit_utilization        0
age                       0
late_30_59_days           0
debt_ratio                0
monthly_income        29731
total_credit_lines        0
late_90_days              0
real_estate_loans         0
late_60_89_days           0
dependents             3924
dtype: int64

In [8]:
# Fill missing monthly_income with median
df['monthly_income'].fillna(df['monthly_income'].median(), inplace=True)

# Fill dependents with median
df['dependents'].fillna(df['dependents'].median(), inplace=True)

# Verify
df.isnull().sum()


C:\Users\Dileep Samaji\AppData\Local\Temp\ipykernel_23024\1594706033.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['monthly_income'].fillna(df['monthly_income'].median(), inplace=True)
C:\Users\Dileep Samaji\AppData\Local\Temp\ipykernel_23024\1594706033.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFr

Unnamed: 0                0
target_default            0
credit_utilization        0
age                       0
late_30_59_days           0
debt_ratio                0
monthly_income        29731
total_credit_lines        0
late_90_days              0
real_estate_loans         0
late_60_89_days           0
dependents             3924
dtype: int64

In [9]:
df.drop(columns=['Unnamed: 0'], errors='ignore', inplace=True)


In [10]:
# Cap extreme values using percentile clipping
for col in df.columns:
    if col != 'target_default':
        lower = df[col].quantile(0.01)
        upper = df[col].quantile(0.99)
        df[col] = np.clip(df[col], lower, upper)

df.describe()


,target_default,credit_utilization,age,late_30_59_days,debt_ratio,monthly_income,total_credit_lines,late_90_days,real_estate_loans,late_60_89_days,dependents
count,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,120269.000000,150000.000000,150000.000000,150000.000000,150000.000000,146076.000000
mean,0.066840,0.320496,52.279260,0.245860,316.548869,6349.112332,8.404000,0.086487,0.992600,0.063180,0.747700
std,0.249746,0.352152,14.668105,0.666815,906.962222,4358.376183,4.946399,0.401673,0.985802,0.290107,1.077946
min,0.000000,0.000000,24.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.029867,41.000000,0.000000,0.175074,3400.000000,5.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.154181,52.000000,0.000000,0.366508,5400.000000,8.000000,0.000000,1.000000,0.000000,0.000000
75%,0.000000,0.559046,63.000000,0.000000,0.868254,8249.000000,11.000000,0.000000,2.000000,0.000000,1.000000
max,1.000000,1.092956,87.000000,4.000000,4979.040000,25000.000000,24.000000,3.000000,4.000000,2.000000,4.000000


In [11]:
# Total late payments
df['total_late_payments'] = (
    df['late_30_59_days'] +
    df['late_60_89_days'] +
    df['late_90_days']
)

# Debt per income ratio (better signal)
df['debt_income_ratio'] = df['debt_ratio'] / (df['monthly_income'] + 1)


In [12]:
df.head()
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 13 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   target_default       150000 non-null  int64  
 1   credit_utilization   150000 non-null  float64
 2   age                  150000 non-null  int64  
 3   late_30_59_days      150000 non-null  int64  
 4   debt_ratio           150000 non-null  float64
 5   monthly_income       120269 non-null  float64
 6   total_credit_lines   150000 non-null  int64  
 7   late_90_days         150000 non-null  int64  
 8   real_estate_loans    150000 non-null  int64  
 9   late_60_89_days      150000 non-null  int64  
 10  dependents           146076 non-null  float64
 11  total_late_payments  150000 non-null  int64  
 12  debt_income_ratio    120269 non-null  float64
dtypes: float64(5), int64(8)
memory usage: 14.9 MB


In [14]:
df.to_csv("../data/processed/cleaned/credit_data_cleaned.csv", index=False)


### Preprocessing **symbols_valid_meta.csv** dataset

### Load dataset

In [16]:
import os

BASE_DIR = os.path.abspath(".")

file_path = os.path.join(BASE_DIR, "../data/raw/market_data/symbols_valid_meta.csv")

df = pd.read_csv(file_path)

df.head()

,Nasdaq Traded,Symbol,Security Name,Listing Exchange,Market Category,ETF,Round Lot Size,Test Issue,Financial Status,CQS Symbol,NASDAQ Symbol,NextShares
0,Y,A,"Agilent Technologies, Inc. Common Stock",N,,N,100.0,N,NaN,A,A,N
1,Y,AA,Alcoa Corporation Common Stock,N,,N,100.0,N,NaN,AA,AA,N
2,Y,AAAU,Perth Mint Physical Gold ETF,P,,Y,100.0,N,NaN,AAAU,AAAU,N
3,Y,AACG,ATA Creativity Global - American Depositary Sh...,Q,G,N,100.0,N,N,NaN,AACG,N
4,Y,AADR,AdvisorShares Dorsey Wright ADR ETF,P,,Y,100.0,N,NaN,AADR,AADR,N


In [17]:
df.info()
df.describe(include='all')
df.isnull().sum()


<class 'pandas.DataFrame'>
RangeIndex: 8049 entries, 0 to 8048
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0       Nasdaq Traded  8049 non-null   str    
 1   Symbol             8049 non-null   str    
 2   Security Name      8049 non-null   str    
 3   Listing Exchange   8049 non-null   str    
 4   Market Category    8049 non-null   str    
 5   ETF                8049 non-null   str    
 6   Round Lot Size     8049 non-null   float64
 7   Test Issue         8049 non-null   str    
 8   Financial Status   3383 non-null   str    
 9   CQS Symbol         4666 non-null   str    
 10  NASDAQ Symbol      8049 non-null   str    
 11  NextShares         8049 non-null   str    
dtypes: float64(1), str(11)
memory usage: 1.2 MB


    Nasdaq Traded       0
Symbol                  0
Security Name           0
Listing Exchange        0
Market Category         0
ETF                     0
Round Lot Size          0
Test Issue              0
Financial Status     4666
CQS Symbol           3383
NASDAQ Symbol           0
NextShares              0
dtype: int64

In [18]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df.columns


Index(['nasdaq_traded', 'symbol', 'security_name', 'listing_exchange',
       'market_category', 'etf', 'round_lot_size', 'test_issue',
       'financial_status', 'cqs_symbol', 'nasdaq_symbol', 'nextshares'],
      dtype='str')

In [19]:
drop_cols = [
    "cqs_symbol",
    "nasdaq_symbol",
    "test_issue",
    "nextshares"
]

df.drop(columns=drop_cols, inplace=True, errors='ignore')


In [20]:
# Replace blanks and spaces with NaN
df.replace([" ", ""], np.nan, inplace=True)

df.isnull().sum()


nasdaq_traded          0
symbol                 0
security_name          0
listing_exchange       0
market_category     4666
etf                    0
round_lot_size         0
financial_status    4666
dtype: int64

In [21]:
# Numeric conversion
df["round_lot_size"] = pd.to_numeric(df["round_lot_size"], errors="coerce")

# Boolean-like columns
binary_map = {"Y": 1, "N": 0}

for col in ["nasdaq_traded", "etf"]:
    df[col] = df[col].map(binary_map)


In [23]:
df.drop_duplicates(inplace=True)
# Keep only actively traded stocks
df = df[df["nasdaq_traded"] == 1]



In [24]:
# Extract clean company name (remove extra text)
df["clean_name"] = df["security_name"].str.split("-").str[0].str.strip()

# Symbol length feature
df["symbol_length"] = df["symbol"].apply(len)


In [25]:
df.info()
df.head()


<class 'pandas.DataFrame'>
RangeIndex: 8049 entries, 0 to 8048
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   nasdaq_traded     8049 non-null   int64  
 1   symbol            8049 non-null   str    
 2   security_name     8049 non-null   str    
 3   listing_exchange  8049 non-null   str    
 4   market_category   3383 non-null   str    
 5   etf               8049 non-null   int64  
 6   round_lot_size    8049 non-null   float64
 7   financial_status  3383 non-null   str    
 8   clean_name        8049 non-null   object 
 9   symbol_length     8049 non-null   int64  
dtypes: float64(1), int64(3), object(1), str(5)
memory usage: 1009.5+ KB


,nasdaq_traded,symbol,security_name,listing_exchange,market_category,etf,round_lot_size,financial_status,clean_name,symbol_length
0,1,A,"Agilent Technologies, Inc. Common Stock",N,NaN,0,100.0,NaN,"Agilent Technologies, Inc. Common Stock",1
1,1,AA,Alcoa Corporation Common Stock,N,NaN,0,100.0,NaN,Alcoa Corporation Common Stock,2
2,1,AAAU,Perth Mint Physical Gold ETF,P,NaN,1,100.0,NaN,Perth Mint Physical Gold ETF,4
3,1,AACG,ATA Creativity Global - American Depositary Sh...,Q,G,0,100.0,N,ATA Creativity Global,4
4,1,AADR,AdvisorShares Dorsey Wright ADR ETF,P,NaN,1,100.0,NaN,AdvisorShares Dorsey Wright ADR ETF,4


In [26]:
output_path = "../data/processed/cleaned/symbols_valid_meta_cleaned.csv"

df.to_csv(output_path, index=False)

print("Saved to:", output_path)


Saved to: ../data/processed/cleaned/symbols_valid_meta_cleaned.csv
